# Notebook 01 — Imbalance Characterization (Phase 1)
## IDS Imbalance Study | RO2 — Track A

**Purpose:**  
Produce a quantitative *Imbalance Profile* for each of the seven IDS benchmark datasets.  
Each profile is a structured fingerprint covering:
- Distribution metrics (IR, Shannon Entropy, Gini Impurity)
- Minority class characterization (within-class variance, Fisher Discriminant Ratio, temporal spread)

**Outputs (saved to `../outputs/phase1/`):**
- `imbalance_profiles.csv` — one row per dataset, all metrics
- `class_distributions/` — per-dataset class count tables
- `minority_profiles/` — per-dataset per-class minority characterization tables
- `figures/` — distribution bar charts, entropy heatmap, FDR heatmaps

**Datasets covered:**  
NSL-KDD · UNSW-NB15 · CIC-IDS2017 · HIKARI-2021 · BoT-IoT · ToN-IoT · CSE-CIC-IDS2018

**Note on loaders:**  
Loaders and leakage-corrected column lists are ported from the IDS-ReComm project.  
Update `DATA_ROOT` in Cell 2 to point to your dataset directory on HPC or local machine.

---
**Phase 2 scaffold** is appended at the bottom of this notebook (clearly marked).  
Phase 2 experiments are not executed here — they depend on Phase 1 outputs.


## Cell 1 — Imports and Global Settings

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — safe for HPC
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif

warnings.filterwarnings('ignore')
np.random.seed(42)

print("Imports OK")

## Cell 2 — Configuration

In [ ]:
# ── PATH CONFIGURATION ──────────────────────────────────────────────────────
# Update DATA_ROOT to your dataset directory.
# HPC example:   DATA_ROOT = Path('/scratch/ifrah/datasets')
# Local example: DATA_ROOT = Path('D:/Datasets')
DATA_ROOT = Path('/path/to/your/datasets')  # <-- UPDATE THIS

OUTPUT_ROOT = Path('../outputs/phase1')
for subdir in ['class_distributions', 'minority_profiles', 'figures']:
    (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)

# ── DATASET FILE PATHS ───────────────────────────────────────────────────────
# Each entry: dataset_name -> dict with file paths and label column.
# CIC-IDS2017 and CSE-CIC-IDS2018 are multi-file; list all CSVs.
DATASET_CONFIG = {
    'NSL-KDD': {
        'files': [DATA_ROOT / 'NSL-KDD' / 'KDDTrain+.txt'],
        'label_col': 'label',
        'has_header': False,
        'separator': ',',
    },
    'UNSW-NB15': {
        'files': [DATA_ROOT / 'UNSW-NB15' / 'UNSW_NB15_training-set.csv',
                  DATA_ROOT / 'UNSW-NB15' / 'UNSW_NB15_testing-set.csv'],
        'label_col': 'label',
        'has_header': True,
        'separator': ',',
    },
    'CIC-IDS2017': {
        'files': sorted((DATA_ROOT / 'CIC-IDS2017').glob('*.csv')),
        'label_col': ' Label',
        'has_header': True,
        'separator': ',',
    },
    'HIKARI-2021': {
        'files': [DATA_ROOT / 'HIKARI-2021' / 'HIKARI_2021.csv'],
        'label_col': 'Attack',
        'has_header': True,
        'separator': ',',
    },
    'BoT-IoT': {
        'files': sorted((DATA_ROOT / 'BoT-IoT').glob('*.csv')),
        'label_col': 'category',
        'has_header': True,
        'separator': ',',
    },
    'ToN-IoT': {
        'files': [DATA_ROOT / 'ToN-IoT' / 'Train_Test_Network.csv'],
        'label_col': 'type',
        'has_header': True,
        'separator': ',',
    },
    'CSE-CIC-IDS2018': {
        'files': sorted((DATA_ROOT / 'CSE-CIC-IDS2018').glob('*.csv')),
        'label_col': 'Label',
        'has_header': True,
        'separator': ',',
    },
    'CICIoT2023': {
        # Use the MERGED_CSV folder — 63 files, all labels present in each file.
        # Features are packet-window aggregates (window=10 or 100 packets),
        # not pure flow-level. Note this when comparing profiles across datasets.
        'files': sorted((DATA_ROOT / 'CICIoT2023' / 'MERGED_CSV').glob('*.csv')),
        'label_col': 'Label',
        'has_header': True,
        'separator': ',',
    },
}

# ── NSL-KDD COLUMN NAMES (no header in raw file) ────────────────────────────
NSL_KDD_COLS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

# ── LEAKAGE-CORRECTED COLUMN DROP LISTS (ported from IDS-ReComm) ────────────
# Columns identified as leakage sources — dropped before any analysis.
LEAKAGE_COLS = {
    'NSL-KDD':        [],  # No leakage columns identified in IDS-ReComm
    'UNSW-NB15':      ['id', 'attack_cat'],
    'CIC-IDS2017':    ['Flow ID', ' Source IP', ' Source Port',
                       ' Destination IP', ' Destination Port', ' Timestamp'],
    'HIKARI-2021':    ['uid', 'originh', 'originp', 'resph', 'respp'],
    'BoT-IoT':        ['pkSeqID', 'stime', 'ltime', 'srcip', 'dstip',
                       'srcport', 'dstport', 'seq', 'attack'],
    'ToN-IoT':        ['ts', 'src_ip', 'dst_ip', 'src_port', 'dst_port'],
    'CSE-CIC-IDS2018':['Timestamp', 'Dst Port'],
    # CICIoT2023: no leakage columns identified in the MERGED_CSV feature set.
    # Features are aggregate statistics over packet windows — no raw IPs/ports/timestamps.
    'CICIoT2023':   [],
}

# ── STANDARDIZED LABEL TAXONOMY (RO1 mapping) ────────────────────────────────
# Maps each dataset's raw label values to a standardized category.
# 'normal' maps to 'Benign'; everything else maps to an attack family.
# Extend as needed when loading each dataset.
LABEL_TAXONOMY = {
    'NSL-KDD': {
        'normal': 'Benign',
        'neptune': 'DoS', 'smurf': 'DoS', 'pod': 'DoS', 'teardrop': 'DoS',
        'land': 'DoS', 'back': 'DoS', 'apache2': 'DoS', 'udpstorm': 'DoS',
        'processtable': 'DoS', 'mailbomb': 'DoS',
        'ipsweep': 'Reconnaissance', 'portsweep': 'Reconnaissance',
        'nmap': 'Reconnaissance', 'satan': 'Reconnaissance',
        'mscan': 'Reconnaissance', 'saint': 'Reconnaissance',
        'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L',
        'multihop': 'R2L', 'named': 'R2L', 'phf': 'R2L', 'sendmail': 'R2L',
        'snmpgetattack': 'R2L', 'snmpguess': 'R2L', 'spy': 'R2L',
        'warezclient': 'R2L', 'warezmaster': 'R2L', 'worm': 'R2L', 'xlock': 'R2L',
        'xsnoop': 'R2L', 'httptunnel': 'R2L',
        'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R',
        'ps': 'U2R', 'rootkit': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R',
    },
    'UNSW-NB15': {
        # Named labels (training/testing CSVs with attack_cat column removed)
        'Normal': 'Benign', 'normal': 'Benign',
        'Fuzzers': 'Fuzzing', 'Analysis': 'Reconnaissance',
        'Backdoor': 'Backdoor', 'DoS': 'DoS', 'Exploits': 'Exploitation',
        'Generic': 'Generic', 'Reconnaissance': 'Reconnaissance',
        'Shellcode': 'Shellcode', 'Worms': 'Worm',
        # Binary integer labels (present in some UNSW-NB15 file variants)
        # 0 = Normal/Benign, 1 = Attack (generic)
        '0': 'Benign', '1': 'Attack',
        0: 'Benign', 1: 'Attack',
    },
    'CIC-IDS2017': {
        'BENIGN': 'Benign',
        'FTP-Patator': 'BruteForce', 'SSH-Patator': 'BruteForce',
        'DoS slowloris': 'DoS', 'DoS Slowhttptest': 'DoS',
        'DoS Hulk': 'DoS', 'DoS GoldenEye': 'DoS', 'Heartbleed': 'Exploitation',
        # Correct UTF-8 em-dash form
        'Web Attack – Brute Force': 'WebAttack',
        'Web Attack – XSS': 'WebAttack',
        'Web Attack – Sql Injection': 'WebAttack',
        # Mojibake form (â) produced when CSVs read without encoding='utf-8'
        'Web Attack â Brute Force': 'WebAttack',
        'Web Attack â XSS': 'WebAttack',
        'Web Attack â Sql Injection': 'WebAttack',
        'Infiltration': 'Infiltration',
        'Bot': 'Botnet',
        'DDoS': 'DDoS',
        'PortScan': 'Reconnaissance',
    },
    'HIKARI-2021': {
        'Benign': 'Benign', 'benign': 'Benign',
        'DoS': 'DoS', 'DDoS': 'DDoS',
        'Ransomware': 'Ransomware', 'Phishing': 'Phishing',
        'SQL Injection': 'WebAttack', 'XSS': 'WebAttack',
        'Insider Threats': 'InsiderThreat',
        # Variants present in actual HIKARI-2021 files
        'Bruteforce-XML': 'BruteForce',
        'Bruteforce': 'BruteForce',
        'Background': 'Background',   # non-attack background traffic
        'Probing': 'Reconnaissance',
        'XMRIGCC CryptoMiner': 'CryptoMining',
    },
    'BoT-IoT': {
        'Normal': 'Benign', 'normal': 'Benign',
        'DDoS': 'DDoS', 'DoS': 'DoS',
        'Reconnaissance': 'Reconnaissance',
        'Theft': 'DataExfiltration',
    },
    'ToN-IoT': {
        'normal': 'Benign',
        'backdoor': 'Backdoor', 'ddos': 'DDoS', 'dos': 'DoS',
        'injection': 'WebAttack', 'mitm': 'MITM', 'password': 'BruteForce',
        'ransomware': 'Ransomware', 'scanning': 'Reconnaissance',
        'xss': 'WebAttack',
    },
    'CICIoT2023': {
        'BENIGN': 'Benign',
        # DDoS variants
        'DDOS-ICMP_FLOOD': 'DDoS', 'DDOS-UDP_FLOOD': 'DDoS',
        'DDOS-TCP_FLOOD': 'DDoS', 'DDOS-PSHACK_FLOOD': 'DDoS',
        'DDOS-RSTFINFLOOD': 'DDoS', 'DDOS-SYN_FLOOD': 'DDoS',
        'DDOS-SYNONYMOUSIP_FLOOD': 'DDoS', 'DDOS-HTTP_FLOOD': 'DDoS',
        'DDOS-SLOWLORIS': 'DDoS', 'DDOS-ICMP_FRAGMENTATION': 'DDoS',
        'DDOS-ACK_FRAGMENTATION': 'DDoS', 'DDOS-UDP_FRAGMENTATION': 'DDoS',
        # DoS variants
        'DOS-UDP_FLOOD': 'DoS', 'DOS-TCP_FLOOD': 'DoS',
        'DOS-SYN_FLOOD': 'DoS', 'DOS-HTTP_FLOOD': 'DoS',
        # Mirai botnet variants
        'MIRAI-GREETH_FLOOD': 'Botnet', 'MIRAI-UDPPLAIN': 'Botnet',
        'MIRAI-GREIP_FLOOD': 'Botnet',
        # Reconnaissance variants
        'VULNERABILITYSCAN': 'Reconnaissance', 'RECON-HOSTDISCOVERY': 'Reconnaissance',
        'RECON-OSSCAN': 'Reconnaissance', 'RECON-PORTSCAN': 'Reconnaissance',
        'RECON-PINGSWEEP': 'Reconnaissance',
        # Spoofing → MITM (closest RO1 category)
        'MITM-ARPSPOOFING': 'MITM', 'DNS_SPOOFING': 'MITM',
        # Brute force
        'DICTIONARYBRUTEFORCE': 'BruteForce',
        # Web-based attacks
        'SQLINJECTION': 'WebAttack', 'COMMANDINJECTION': 'WebAttack',
        'XSS': 'WebAttack', 'UPLOADING_ATTACK': 'WebAttack',
        'BROWSERHIJACKING': 'WebAttack',
        # Backdoor
        'BACKDOOR_MALWARE': 'Backdoor',
    },
    'CSE-CIC-IDS2018': {
        'Benign': 'Benign', 'benign': 'Benign',
        # Brute force variants
        'FTP-BruteForce': 'BruteForce', 'SSH-BruteForce': 'BruteForce',
        'FTP-Bruteforce': 'BruteForce', 'SSH-Bruteforce': 'BruteForce',
        'Brute Force -Web': 'BruteForce', 'Brute Force -XSS': 'WebAttack',
        # DoS variants — two naming conventions present in raw files
        'DoS-GoldenEye': 'DoS', 'DoS-Slowloris': 'DoS',
        'DoS-SlowHTTPTest': 'DoS', 'DoS-Hulk': 'DoS',
        'DoS attacks-GoldenEye': 'DoS', 'DoS attacks-Slowloris': 'DoS',
        'DoS attacks-SlowHTTPTest': 'DoS', 'DoS attacks-Hulk': 'DoS',
        # Infiltration (note the typo present in raw files)
        'Infilteration': 'Infiltration', 'Infiltration': 'Infiltration',
        # Botnet
        'Bot': 'Botnet',
        # DDoS variants
        'DDoS attacks-LOIC-HTTP': 'DDoS',
        'DDOS attack-HOIC': 'DDoS', 'DDOS attack-LOIC-UDP': 'DDoS',
        # Web attacks
        'SQL Injection': 'WebAttack',
    },
}

print("Configuration loaded.")
print(f"Datasets configured: {list(DATASET_CONFIG.keys())}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")

## Cell 3 — Dataset Loader (ported from IDS-ReComm)

In [ ]:
def load_dataset(name: str, config: dict) -> pd.DataFrame:
    """
    Load a dataset from one or more CSV files.
    Applies leakage column removal and label taxonomy mapping.
    Returns a DataFrame with a clean 'label' column (raw) and
    'label_std' column (standardized taxonomy).
    """
    dfs = []
    for fpath in config['files']:
        if not Path(fpath).exists():
            print(f"  [WARN] File not found, skipping: {fpath}")
            continue
        if name == 'NSL-KDD' and not config['has_header']:
            df = pd.read_csv(fpath, header=None, names=NSL_KDD_COLS,
                             sep=config['separator'])
            df.drop(columns=['difficulty'], inplace=True, errors='ignore')
        else:
            df = pd.read_csv(fpath, sep=config['separator'],
                             low_memory=False)

        # CSE-CIC-IDS2018 CSV artifact: some rows contain the string "Label"
        # as a data value due to file concatenation. Drop them immediately.
        if name == 'CSE-CIC-IDS2018' and config['label_col'] in df.columns:
            artifact_mask = df[config['label_col']].astype(str).str.strip() == 'Label'
            n_artifact = artifact_mask.sum()
            if n_artifact > 0:
                df = df[~artifact_mask]
                print(f"  Dropped {n_artifact} CSV artifact rows (Label==\'Label\')")

        dfs.append(df)

    if not dfs:
        raise FileNotFoundError(f"No files loaded for dataset: {name}")

    df = pd.concat(dfs, ignore_index=True)
    print(f"  Loaded {name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

    # Strip column name whitespace early (CIC datasets)
    df.columns = df.columns.str.strip()

    # Rename label column to 'label' for consistency.
    # If the label_col differs from 'label' and a 'label' column already exists
    # (e.g. ToN-IoT has both 'type' and possibly 'label'), drop the stale one first.
    label_col = config['label_col'].strip()
    if label_col in df.columns and label_col != 'label':
        if 'label' in df.columns:
            df.drop(columns=['label'], inplace=True)
        df.rename(columns={label_col: 'label'}, inplace=True)
    elif label_col not in df.columns and 'label' not in df.columns:
        raise KeyError(f"Label column '{label_col}' not found in {name}. "
                       f"Available columns: {list(df.columns)}")

    # Drop leakage columns
    leakage = [c for c in LEAKAGE_COLS.get(name, []) if c in df.columns]
    if leakage:
        df.drop(columns=leakage, inplace=True)
        print(f"  Dropped {len(leakage)} leakage columns: {leakage}")

    # Ensure label column is string before stripping
    df['label'] = df['label'].astype(str).str.strip()

    # Apply standardized taxonomy mapping
    taxonomy = LABEL_TAXONOMY.get(name, {})
    df['label_std'] = df['label'].map(taxonomy)
    unmapped = df['label_std'].isna().sum()
    if unmapped > 0:
        unmapped_vals = df.loc[df['label_std'].isna(), 'label'].unique()
        print(f"  [WARN] {unmapped:,} rows with unmapped labels: {unmapped_vals}")
        # Fall back to raw label for unmapped values
        df['label_std'] = df['label_std'].fillna(df['label'])

    # Drop rows with inf
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    before = len(df)
    df.dropna(subset=['label'], inplace=True)
    if len(df) < before:
        print(f"  Dropped {before - len(df):,} rows with null labels")

    print(f"  Final shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print(f"  Unique raw labels: {df['label'].nunique()} | "
          f"Standardized: {df['label_std'].nunique()}")
    return df


print("Loader defined.")


## Cell 4 — Imbalance Metric Functions

In [ ]:
def compute_distribution_metrics(label_series: pd.Series) -> dict:
    """
    Compute dataset-level imbalance distribution metrics.

    Returns:
        dict with keys:
          n_classes       : number of unique classes
          n_samples       : total sample count
          majority_class  : name of the largest class
          majority_count  : sample count of majority class
          minority_class  : name of the smallest class
          minority_count  : sample count of minority class
          IR              : Imbalance Ratio (majority / minority)
          shannon_entropy : Shannon entropy of class distribution (nats)
          entropy_norm    : Shannon entropy normalized by log(n_classes)
                            (0 = maximally imbalanced, 1 = perfectly balanced)
          gini_impurity   : Gini impurity of class distribution
          benign_pct      : percentage of samples labelled Benign/Normal
    """
    counts = label_series.value_counts()
    n = len(label_series)
    probs = counts / n

    sh_entropy = scipy_entropy(probs.values)  # natural log base
    max_entropy = np.log(len(counts)) if len(counts) > 1 else 1.0
    entropy_norm = sh_entropy / max_entropy if max_entropy > 0 else 0.0

    gini = 1.0 - np.sum(probs.values ** 2)

    # Benign percentage — look for common benign label names
    benign_keys = {'benign', 'normal', 'Benign', 'Normal', 'BENIGN'}
    benign_count = counts[counts.index.isin(benign_keys)].sum()
    benign_pct = 100.0 * benign_count / n

    return {
        'n_classes':       len(counts),
        'n_samples':       n,
        'majority_class':  counts.index[0],
        'majority_count':  int(counts.iloc[0]),
        'minority_class':  counts.index[-1],
        'minority_count':  int(counts.iloc[-1]),
        'IR':              round(counts.iloc[0] / counts.iloc[-1], 2),
        'shannon_entropy': round(sh_entropy, 4),
        'entropy_norm':    round(entropy_norm, 4),
        'gini_impurity':   round(gini, 4),
        'benign_pct':      round(benign_pct, 2),
    }


def compute_minority_profiles(df: pd.DataFrame,
                               label_col: str = 'label_std',
                               top_n_minority: int = 10) -> pd.DataFrame:
    """
    For each class (sorted by count ascending), compute:
      - count, pct
      - mean_within_class_variance: average variance of numeric features
        within that class (proxy for intra-class spread)
      - fisher_ratio: Fisher Discriminant Ratio vs. all other samples
        (between-class variance / within-class variance, averaged over features)
        Higher = more separable from the rest.

    Returns a DataFrame sorted by count ascending (rarest first).
    Only numeric columns are used for variance/FDR computations.
    """
    counts = df[label_col].value_counts().sort_values(ascending=True)
    n_total = len(df)

    # Select numeric columns using pandas dtype check — more reliable than
    # select_dtypes after a cast, which can silently produce NaN for object cols.
    numeric_cols = [c for c in df.columns
                    if c not in [label_col, 'label']
                    and pd.api.types.is_numeric_dtype(df[c])
                    and df[c].notna().sum() > 0]

    if not numeric_cols:
        print("  [WARN] No numeric columns found for minority profiling.")
        return pd.DataFrame()

    # For very large datasets, compute global stats on a capped sample
    # to avoid materialising the full numeric block in memory.
    MAX_GLOBAL_SAMPLE = 500_000
    if len(df) > MAX_GLOBAL_SAMPLE:
        global_sample = df[numeric_cols].sample(MAX_GLOBAL_SAMPLE, random_state=42)
    else:
        global_sample = df[numeric_cols]

    global_mean = global_sample.mean()
    global_var  = global_sample.var().replace(0, np.nan)  # avoid /0
    del global_sample

    records = []
    for cls, cnt in counts.items():
        mask = df[label_col] == cls
        cls_df = df.loc[mask, numeric_cols]

        # For very small classes, skip if fewer than 2 samples (var undefined)
        if cnt < 2:
            records.append({
                'class':             cls,
                'count':             cnt,
                'pct':               round(100.0 * cnt / n_total, 4),
                'mean_within_var':   float('nan'),
                'mean_fisher_ratio': float('nan'),
            })
            continue

        within_var = cls_df.var()
        mean_within_var = float(within_var.mean())

        cls_mean = cls_df.mean()
        common_cols = cls_mean.index.intersection(global_mean.index)
        fdr_per_feature = ((cls_mean[common_cols] - global_mean[common_cols]) ** 2
                           / global_var[common_cols].replace(np.nan, 1e-9))
        mean_fdr = float(fdr_per_feature.mean())

        records.append({
            'class':                cls,
            'count':                cnt,
            'pct':                  round(100.0 * cnt / n_total, 4),
            'mean_within_var':      round(mean_within_var, 4),
            'mean_fisher_ratio':    round(mean_fdr, 4),
        })

    return pd.DataFrame(records)


def compute_temporal_spread(df: pd.DataFrame,
                             label_col: str = 'label_std',
                             timestamp_col: str = None) -> pd.DataFrame:
    """
    If a timestamp column is available, compute for each class:
      - first_occurrence, last_occurrence
      - temporal_span_pct: fraction of total dataset time span occupied
        (0 = concentrated in one window, 1 = spread across entire capture)

    If no timestamp column is available or found, returns an empty DataFrame.
    Uses row index as a proxy for time order if no timestamp is provided.
    """
    # Try to find a timestamp column if not specified
    if timestamp_col is None:
        ts_candidates = [c for c in df.columns
                         if any(k in c.lower() for k in
                                ['time', 'timestamp', 'ts', 'stime', 'ltime'])]
        timestamp_col = ts_candidates[0] if ts_candidates else None

    if timestamp_col and timestamp_col in df.columns:
        try:
            ts = pd.to_numeric(df[timestamp_col], errors='coerce')
        except Exception:
            ts = None
    else:
        # Use row index as time proxy
        ts = pd.Series(range(len(df)), index=df.index)
        print("  [INFO] No timestamp column found — using row index as temporal proxy.")

    if ts is None or ts.isna().all():
        return pd.DataFrame()

    total_span = ts.max() - ts.min()
    if total_span == 0:
        return pd.DataFrame()

    records = []
    for cls in df[label_col].unique():
        mask = df[label_col] == cls
        cls_ts = ts[mask].dropna()
        if len(cls_ts) == 0:
            continue
        span = cls_ts.max() - cls_ts.min()
        records.append({
            'class':              cls,
            'first_occurrence':   cls_ts.min(),
            'last_occurrence':    cls_ts.max(),
            'temporal_span_pct':  round(100.0 * span / total_span, 2),
        })

    return pd.DataFrame(records).sort_values('temporal_span_pct')


print("Metric functions defined.")

## Cell 5 — Plotting Helpers

In [ ]:
def plot_class_distribution(counts: pd.Series, dataset_name: str,
                             save_dir: Path):
    """
    Horizontal bar chart of class counts (log scale).
    Saved to save_dir / f'{dataset_name}_class_dist.png'
    """
    fig, ax = plt.subplots(figsize=(10, max(4, len(counts) * 0.5)))
    colors = ['#2196F3' if 'benign' in str(c).lower() or 'normal' in str(c).lower()
              else '#E53935' for c in counts.index]
    counts.sort_values().plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.set_xscale('log')
    ax.set_xlabel('Sample Count (log scale)', fontsize=11)
    ax.set_title(f'{dataset_name} — Class Distribution', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    # Annotate with counts
    for i, (idx, val) in enumerate(counts.sort_values().items()):
        ax.text(val * 1.05, i, f'{val:,}', va='center', fontsize=8)
    plt.tight_layout()
    out = save_dir / f'{dataset_name}_class_dist.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Figure saved: {out.name}")


def plot_entropy_heatmap(profiles_df: pd.DataFrame, save_dir: Path):
    """
    Heatmap of key imbalance metrics across all datasets.
    """
    metrics = ['IR', 'entropy_norm', 'gini_impurity', 'benign_pct', 'n_classes']
    available = [m for m in metrics if m in profiles_df.columns]
    plot_df = profiles_df.set_index('dataset')[available].astype(float)

    # Normalize each column to 0-1 for visual comparability
    plot_norm = (plot_df - plot_df.min()) / (plot_df.max() - plot_df.min() + 1e-9)

    fig, ax = plt.subplots(figsize=(10, max(4, len(plot_norm) * 0.7)))
    sns.heatmap(plot_norm, annot=plot_df.round(2), fmt='g',
                cmap='RdYlGn_r', linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Normalized value (0=best, 1=worst)'})
    ax.set_title('Imbalance Profile Heatmap — All Datasets', fontsize=13,
                 fontweight='bold')
    ax.set_xlabel('')
    plt.tight_layout()
    out = save_dir / 'imbalance_heatmap_all_datasets.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Heatmap saved: {out.name}")


def plot_fdr_heatmap(minority_df: pd.DataFrame, dataset_name: str,
                     save_dir: Path):
    """
    Bar chart of Fisher Discriminant Ratio per class for a single dataset.
    """
    if minority_df.empty:
        return
    fig, ax = plt.subplots(figsize=(10, max(4, len(minority_df) * 0.5)))
    minority_df_sorted = minority_df.sort_values('mean_fisher_ratio')
    bars = ax.barh(minority_df_sorted['class'], minority_df_sorted['mean_fisher_ratio'],
                   color='#7B1FA2', edgecolor='white')
    ax.set_xlabel('Mean Fisher Discriminant Ratio', fontsize=11)
    ax.set_title(f'{dataset_name} — Separability by Class (FDR)', fontsize=13,
                 fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    out = save_dir / f'{dataset_name}_fdr.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  FDR figure saved: {out.name}")


print("Plotting helpers defined.")

## Cell 6 — Main Phase 1 Loop
Iterates over all configured datasets, computes full imbalance profile, saves CSVs and figures.

In [ ]:
all_profiles   = []   # dataset-level summary rows
all_minority   = {}   # per-dataset minority profile DataFrames
all_temporal   = {}   # per-dataset temporal spread DataFrames
load_errors    = []   # datasets that failed to load

for ds_name, ds_config in DATASET_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"Processing: {ds_name}")
    print(f"{'='*60}")

    # ── Load ────────────────────────────────────────────────────────────────
    try:
        df = load_dataset(ds_name, ds_config)
    except Exception as e:
        print(f"  [ERROR] Could not load {ds_name}: {e}")
        load_errors.append(ds_name)
        continue

    # ── Distribution metrics (using standardized labels) ─────────────────────
    print("\n  Computing distribution metrics...")
    dist_metrics = compute_distribution_metrics(df['label_std'])
    dist_metrics['dataset'] = ds_name
    all_profiles.append(dist_metrics)

    # Save class counts table
    counts_df = df['label_std'].value_counts().reset_index()
    counts_df.columns = ['class', 'count']
    counts_df['pct'] = (100.0 * counts_df['count'] / len(df)).round(4)
    counts_out = OUTPUT_ROOT / 'class_distributions' / f'{ds_name}_class_counts.csv'
    counts_df.to_csv(counts_out, index=False)
    print(f"  Class counts saved: {counts_out.name}")

    # Plot class distribution
    plot_class_distribution(
        df['label_std'].value_counts(),
        ds_name,
        OUTPUT_ROOT / 'figures'
    )

    # Print summary
    print(f"\n  Distribution Summary:")
    for k, v in dist_metrics.items():
        if k != 'dataset':
            print(f"    {k:25s}: {v}")

    # ── Minority class profiles ──────────────────────────────────────────────
    print("\n  Computing minority class profiles (FDR + within-class variance)...")
    try:
        min_df = compute_minority_profiles(df, label_col='label_std')
        all_minority[ds_name] = min_df
        min_out = OUTPUT_ROOT / 'minority_profiles' / f'{ds_name}_minority_profile.csv'
        min_df.to_csv(min_out, index=False)
        print(f"  Minority profiles saved: {min_out.name}")
        print(min_df.to_string(index=False))

        # Plot FDR
        plot_fdr_heatmap(min_df, ds_name, OUTPUT_ROOT / 'figures')
    except Exception as e:
        print(f"  [WARN] Minority profile failed: {e}")

    # ── Temporal spread ──────────────────────────────────────────────────────
    print("\n  Computing temporal spread...")
    try:
        temp_df = compute_temporal_spread(df, label_col='label_std')
        if not temp_df.empty:
            all_temporal[ds_name] = temp_df
            temp_out = OUTPUT_ROOT / 'class_distributions' / f'{ds_name}_temporal_spread.csv'
            temp_df.to_csv(temp_out, index=False)
            print(f"  Temporal spread saved: {temp_out.name}")
            print(temp_df.to_string(index=False))
    except Exception as e:
        print(f"  [WARN] Temporal spread failed: {e}")

    # Free memory before next dataset
    del df

print(f"\n{'='*60}")
print("Phase 1 loop complete.")
if load_errors:
    print(f"Datasets that failed to load: {load_errors}")

## Cell 7 — Aggregate Imbalance Profile Table + Cross-Dataset Heatmap

In [ ]:
if not all_profiles:
    print("No profiles to aggregate — check dataset paths.")
else:
    profiles_df = pd.DataFrame(all_profiles)
    # Reorder columns for readability
    col_order = ['dataset', 'n_samples', 'n_classes', 'majority_class',
                 'majority_count', 'minority_class', 'minority_count',
                 'IR', 'shannon_entropy', 'entropy_norm',
                 'gini_impurity', 'benign_pct']
    profiles_df = profiles_df[[c for c in col_order if c in profiles_df.columns]]

    # Save master profile CSV
    profile_out = OUTPUT_ROOT / 'imbalance_profiles.csv'
    profiles_df.to_csv(profile_out, index=False)
    print(f"Master imbalance profile saved: {profile_out}")
    print("\n" + profiles_df.to_string(index=False))

    # Cross-dataset heatmap
    plot_entropy_heatmap(profiles_df, OUTPUT_ROOT / 'figures')

## Cell 8 — Phase 1 Interpretation Notes

Run this cell after Cell 7 to auto-generate a plain-language interpretation of the imbalance profiles — ready to paste into your paper draft.

In [ ]:
def interpret_profile(row: pd.Series) -> str:
    """
    Generate a one-paragraph plain-language interpretation of a dataset's
    imbalance profile for use in paper writing.
    """
    lines = []
    lines.append(f"**{row['dataset']}** ({row['n_samples']:,} samples, "
                 f"{row['n_classes']} classes):")

    # IR severity
    ir = row['IR']
    if ir < 10:
        ir_desc = "mild"
    elif ir < 100:
        ir_desc = "moderate"
    elif ir < 1000:
        ir_desc = "severe"
    else:
        ir_desc = "extreme"
    lines.append(f"  IR={ir} ({ir_desc} imbalance). "
                 f"Majority class: '{row['majority_class']}' "
                 f"({row['majority_count']:,} samples). "
                 f"Rarest class: '{row['minority_class']}' "
                 f"({row['minority_count']:,} samples).")

    # Entropy
    en = row['entropy_norm']
    en_desc = "near-uniform" if en > 0.8 else ("moderately spread" if en > 0.5
              else ("concentrated" if en > 0.2 else "highly concentrated"))
    lines.append(f"  Normalized entropy={en} → distribution is {en_desc}.")

    # Benign pct
    bp = row['benign_pct']
    if bp > 80:
        lines.append(f"  Benign traffic dominates ({bp}% of samples), "
                     f"typical of real-world capture but risks masking rare attacks.")
    elif bp < 20:
        lines.append(f"  Low benign proportion ({bp}%), suggesting a "
                     f"capture focused on attack traffic — reduced ecological validity.")
    else:
        lines.append(f"  Benign proportion is {bp}% — reasonable balance "
                     f"between normal and attack traffic.")

    return "\n".join(lines)


if 'profiles_df' in dir() and not profiles_df.empty:
    print("=" * 70)
    print("AUTO-GENERATED INTERPRETATION NOTES")
    print("=" * 70)
    for _, row in profiles_df.iterrows():
        print()
        print(interpret_profile(row))
    print()
    print("=" * 70)
else:
    print("Run Cell 7 first.")

---
---
# PHASE 2 SCAFFOLD — Evaluation Distortion Study

> **Status:** Scaffold only. Cells below define the full experimental design  
> and all helper functions but do **not** execute experiments.  
> Execute after Phase 1 outputs are reviewed and `imbalance_profiles.csv` is finalized.

**Research question:**  
Given a known imbalance profile, how much does it distort what we conclude about model performance — and which metrics are most sensitive?

**Design summary:**
1. For each dataset, construct 5 controlled imbalance scenarios via stratified subsampling
2. Train RF + XGBoost on each scenario (fast, strong baselines)
3. Measure macro-F1, macro PR-AUC, MCC, per-class recall — and compute Metric Sensitivity Index
4. Assess cross-dataset generalization: train on Dataset A, test on Dataset B
5. Derive IR ceiling and minimum minority class size thresholds → RO3 design principles


## [P2-Cell 1] Phase 2 — Additional Imports and Config (Scaffold)

In [ ]:
# ── Phase 2 imports ──────────────────────────────────────────────────────────
# Uncomment when ready to run Phase 2.

# from sklearn.ensemble import RandomForestClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import StratifiedShuffleSplit
# from sklearn.metrics import (f1_score, matthews_corrcoef,
#                               average_precision_score,
#                               classification_report)
# from sklearn.pipeline import Pipeline
# import xgboost as xgb

# ── Phase 2 output directories ───────────────────────────────────────────────
# P2_OUTPUT = Path('../outputs/phase2')
# for subdir in ['scenario_results', 'cross_dataset', 'figures', 'msi']:
#     (P2_OUTPUT / subdir).mkdir(parents=True, exist_ok=True)

# ── Imbalance scenario definitions ───────────────────────────────────────────
# Five scenarios per dataset.
# Scenario 0 = native (no subsampling)
# Scenarios 1-4 = progressively corrected IR targets via minority oversampling
# or majority undersampling (majority class only, to preserve minority samples).
#
# IR_TARGETS = [None, 100, 20, 5, 1]
# SCENARIO_LABELS = ['Native', 'IR=100', 'IR=20', 'IR=5', 'IR=1 (Balanced)']
#
# Models for Phase 2
# # Stored as callables so each dataset gets a fresh instance.
# # This avoids XGBoost carrying over num_class from a prior fit.
# P2_MODELS = {
#     'RandomForest': lambda: RandomForestClassifier(
#                         n_estimators=100, n_jobs=-1,
#                         random_state=42, class_weight='balanced'),
#     'XGBoost':      lambda: xgb.XGBClassifier(
#                         n_estimators=100, n_jobs=-1,
#                         random_state=42, eval_metric='mlogloss',
#                         use_label_encoder=False),
# }
# N_REPEATS = 5  # repeated runs for variance estimation

print("Phase 2 scaffold config cell — uncomment to activate.")

## [P2-Cell 2] Controlled Imbalance Scenario Generator (Scaffold)

In [ ]:
# def create_imbalance_scenario(df: pd.DataFrame,
#                                label_col: str,
#                                target_ir: float = None,
#                                random_state: int = 42) -> pd.DataFrame:
#     """
#     Create a controlled imbalance scenario by undersampling the majority class.
#     Minority classes are always kept in full.
#
#     Args:
#         df         : Input DataFrame
#         label_col  : Column name for class labels
#         target_ir  : Target Imbalance Ratio (majority/minority).
#                      If None, returns df unchanged (native scenario).
#         random_state: Reproducibility seed
#
#     Returns:
#         Subsampled DataFrame with the target IR.
#         If the native IR is already <= target_ir, returns df unchanged.
#     """
#     if target_ir is None:
#         return df.copy()
#
#     counts = df[label_col].value_counts()
#     minority_count = counts.iloc[-1]  # smallest class
#     majority_class = counts.index[0]
#     native_ir = counts.iloc[0] / minority_count
#
#     if native_ir <= target_ir:
#         print(f"    Native IR ({native_ir:.1f}) already <= target ({target_ir}). "
#               "Returning unchanged.")
#         return df.copy()
#
#     target_majority_count = int(minority_count * target_ir)
#     maj_df = df[df[label_col] == majority_class].sample(
#         n=target_majority_count, random_state=random_state)
#     rest_df = df[df[label_col] != majority_class]
#     return pd.concat([maj_df, rest_df], ignore_index=True).sample(
#         frac=1, random_state=random_state)  # shuffle

print("Phase 2 scenario generator scaffold defined (commented out).")

## [P2-Cell 3] Metric Sensitivity Index Function (Scaffold)

In [ ]:
# def compute_metric_sensitivity_index(results_df: pd.DataFrame,
#                                       metric_col: str = 'macro_f1',
#                                       ir_col: str = 'IR') -> float:
#     """
#     Compute the Metric Sensitivity Index (MSI) for a given metric.
#
#     MSI = mean absolute change in metric per unit change in log(IR).
#     Higher MSI means the metric is highly sensitive to imbalance changes —
#     i.e., reported values in the literature are less reliable for this dataset.
#
#     Args:
#         results_df : DataFrame with one row per scenario,
#                      must contain ir_col and metric_col.
#         metric_col : Column name of the metric to analyze.
#         ir_col     : Column name of the IR value for each scenario.
#
#     Returns:
#         MSI (float)
#     """
#     df_sorted = results_df.sort_values(ir_col).copy()
#     log_ir = np.log(df_sorted[ir_col].replace(0, 1e-9))
#     metric_vals = df_sorted[metric_col].values
#
#     if len(log_ir) < 2:
#         return 0.0
#
#     # Finite differences: delta_metric / delta_log_IR
#     delta_metric = np.abs(np.diff(metric_vals))
#     delta_log_ir = np.abs(np.diff(log_ir.values))
#     sensitivities = delta_metric / (delta_log_ir + 1e-9)
#     return float(np.mean(sensitivities))

print("Phase 2 MSI scaffold defined (commented out).")

## [P2-Cell 4] Phase 2 Main Experiment Loop (Scaffold)

Full design:
- For each dataset × scenario × model × repeat:
  - Stratified 80/20 train/test split
  - StandardScaler fit on train only
  - Train model, predict on test
  - Record: macro-F1, macro PR-AUC, MCC, per-class recall, train time, test time
- Aggregate: mean ± std across repeats
- Compute MSI per dataset per metric

In [ ]:
# # ── Phase 2 Main Loop ────────────────────────────────────────────────────────
# # NOTE: This loop is compute-intensive. Run on HPC with screen.
# # Estimated runtime: 2-6 hours depending on dataset sizes.
#
# import time
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import StratifiedShuffleSplit
# from sklearn.metrics import f1_score, matthews_corrcoef, classification_report
#
# all_p2_results = []
#
# for ds_name, ds_config in DATASET_CONFIG.items():
#     if ds_name in load_errors:
#         continue
#     print(f"\n{'='*60}\nPhase 2: {ds_name}\n{'='*60}")
#
#     df = load_dataset(ds_name, ds_config)
#
#     # Feature matrix: numeric only, drop label columns
#     drop_cols = ['label', 'label_std']
#     feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
#                     if c not in drop_cols]
#     X = df[feature_cols].fillna(0).values
#     y = df['label_std'].values
#
#     le = LabelEncoder()
#     y_enc = le.fit_transform(y)
#
#     for scenario_label, target_ir in zip(SCENARIO_LABELS, IR_TARGETS):
#         print(f"  Scenario: {scenario_label}")
#         df_scenario = create_imbalance_scenario(df, 'label_std', target_ir)
#         X_s = df_scenario[feature_cols].fillna(0).values
#         y_s = le.transform(df_scenario['label_std'].values)
#
#         actual_ir = compute_distribution_metrics(df_scenario['label_std'])['IR']
#
#         for model_name in P2_MODELS.keys():
#             # Instantiate a fresh model for each dataset×scenario to avoid
#             # stale XGBoost internal state (e.g. cached num_class from prior fit).
#             model = P2_MODELS[model_name]()
#             f1_scores, mcc_scores = [], []
#             for rep in range(N_REPEATS):
#                 sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2,
#                                              random_state=rep)
#                 train_idx, test_idx = next(sss.split(X_s, y_s))
#                 X_tr, X_te = X_s[train_idx], X_s[test_idx]
#                 y_tr, y_te = y_s[train_idx], y_s[test_idx]
#
#                 scaler = StandardScaler()
#                 X_tr = scaler.fit_transform(X_tr)
#                 X_te = scaler.transform(X_te)
#
#                 t0 = time.time()
#                 model.fit(X_tr, y_tr)
#                 train_time = time.time() - t0
#
#                 t0 = time.time()
#                 y_pred = model.predict(X_te)
#                 test_time = time.time() - t0
#
#                 f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
#                 mcc = matthews_corrcoef(y_te, y_pred)
#                 f1_scores.append(f1)
#                 mcc_scores.append(mcc)
#
#                 all_p2_results.append({
#                     'dataset':      ds_name,
#                     'scenario':     scenario_label,
#                     'IR':           actual_ir,
#                     'model':        model_name,
#                     'repeat':       rep,
#                     'macro_f1':     round(f1, 4),
#                     'mcc':          round(mcc, 4),
#                     'train_time_s': round(train_time, 3),
#                     'test_time_s':  round(test_time, 3),
#                 })
#
#             print(f"    {model_name}: macro-F1={np.mean(f1_scores):.4f} "
#                   f"±{np.std(f1_scores):.4f} | MCC={np.mean(mcc_scores):.4f}")
#
#     del df
#
# p2_results_df = pd.DataFrame(all_p2_results)
# p2_out = P2_OUTPUT / 'scenario_results' / 'all_scenario_results.csv'
# p2_results_df.to_csv(p2_out, index=False)
# print(f"\nPhase 2 results saved: {p2_out}")

print("Phase 2 main loop scaffold defined (commented out).")

## [P2-Cell 5] MSI Computation + Design Principles Derivation (Scaffold)

In [ ]:
# # Compute MSI for macro-F1 and MCC per dataset
# msi_records = []
# for ds_name in p2_results_df['dataset'].unique():
#     for model_name in p2_results_df['model'].unique():
#         subset = (p2_results_df
#                   [(p2_results_df['dataset'] == ds_name) &
#                    (p2_results_df['model'] == model_name)]
#                   .groupby('scenario')
#                   .agg(IR=('IR', 'mean'),
#                        macro_f1=('macro_f1', 'mean'),
#                        mcc=('mcc', 'mean'))
#                   .reset_index())
#
#         msi_f1  = compute_metric_sensitivity_index(subset, 'macro_f1', 'IR')
#         msi_mcc = compute_metric_sensitivity_index(subset, 'mcc', 'IR')
#
#         msi_records.append({
#             'dataset': ds_name,
#             'model':   model_name,
#             'MSI_macro_f1': round(msi_f1, 4),
#             'MSI_mcc':      round(msi_mcc, 4),
#         })
#
# msi_df = pd.DataFrame(msi_records)
# msi_out = P2_OUTPUT / 'msi' / 'msi_results.csv'
# msi_df.to_csv(msi_out, index=False)
# print(msi_df.to_string(index=False))
#
# # ── Design Principle derivation ───────────────────────────────────────────
# # After Phase 2 is run, use msi_df and p2_results_df to derive:
# #
# # Principle 1 — IR ceiling:
# #   Find the IR threshold above which MSI_macro_f1 exceeds a chosen
# #   instability threshold (e.g., 0.05 F1 per log-IR unit).
# #   This becomes the maximum IR allowed in RO3 dataset design.
# #
# # Principle 2 — Minimum minority class size:
# #   From minority_profiles, find the count below which mean_fisher_ratio
# #   drops sharply — indicating the class becomes unlearnable.
# #   This sets the minimum capture requirement per attack type in RO3.
# #
# # Principle 3 — Target distribution:
# #   From cross-dataset generalization results (P2-Cell 6),
# #   identify which training IR produces the best held-out generalization.
# #   Propose this as the target distribution for RO3.

print("Phase 2 MSI + design principles scaffold defined (commented out).")

## [P2-Cell 6] Cross-Dataset Generalization (Scaffold)

Train on Dataset A (native IR), test on Dataset B — using only the shared standardized label set.
Measures whether balancing at training time improves generalization to unseen data distributions.

In [ ]:
# # Cross-dataset generalization scaffold
# # Pairs to evaluate — choose datasets that share at least 3 standardized labels
# CROSS_PAIRS = [
#     ('NSL-KDD',    'UNSW-NB15'),
#     ('CIC-IDS2017','CSE-CIC-IDS2018'),
#     ('BoT-IoT',    'ToN-IoT'),
#     ('UNSW-NB15',  'CIC-IDS2017'),
# ]
#
# cross_results = []
#
# for train_ds, test_ds in CROSS_PAIRS:
#     print(f"\nCross: train={train_ds} → test={test_ds}")
#     # Load both datasets
#     df_train = load_dataset(train_ds, DATASET_CONFIG[train_ds])
#     df_test  = load_dataset(test_ds,  DATASET_CONFIG[test_ds])
#
#     # Find shared standardized labels
#     shared_labels = (set(df_train['label_std'].unique()) &
#                      set(df_test['label_std'].unique()))
#     if len(shared_labels) < 3:
#         print(f"  Skipping — fewer than 3 shared labels: {shared_labels}")
#         continue
#     print(f"  Shared labels ({len(shared_labels)}): {shared_labels}")
#
#     # Filter both to shared labels
#     df_train = df_train[df_train['label_std'].isin(shared_labels)]
#     df_test  = df_test[df_test['label_std'].isin(shared_labels)]
#
#     # Align feature columns
#     drop_cols = ['label', 'label_std']
#     train_feats = set(df_train.select_dtypes(include=[np.number]).columns) - set(drop_cols)
#     test_feats  = set(df_test.select_dtypes(include=[np.number]).columns)  - set(drop_cols)
#     shared_feats = sorted(train_feats & test_feats)
#     if len(shared_feats) < 5:
#         print(f"  Skipping — fewer than 5 shared features.")
#         continue
#
#     X_train = df_train[shared_feats].fillna(0).values
#     X_test  = df_test[shared_feats].fillna(0).values
#
#     le_cross = LabelEncoder().fit(list(shared_labels))
#     y_train = le_cross.transform(df_train['label_std'])
#     y_test  = le_cross.transform(df_test['label_std'])
#
#     for scenario_label, target_ir in zip(SCENARIO_LABELS[:3], IR_TARGETS[:3]):
#         df_tr_s = create_imbalance_scenario(df_train, 'label_std', target_ir)
#         X_tr_s = df_tr_s[shared_feats].fillna(0).values
#         y_tr_s = le_cross.transform(df_tr_s['label_std'])
#
#         scaler = StandardScaler()
#         X_tr_s = scaler.fit_transform(X_tr_s)
#         X_te_s = scaler.transform(X_test)
#
#         rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
#         rf.fit(X_tr_s, y_tr_s)
#         y_pred = rf.predict(X_te_s)
#         f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
#         mcc = matthews_corrcoef(y_test, y_pred)
#         print(f"  {scenario_label}: macro-F1={f1:.4f} | MCC={mcc:.4f}")
#
#         cross_results.append({
#             'train_dataset': train_ds,
#             'test_dataset':  test_ds,
#             'scenario':      scenario_label,
#             'n_shared_labels': len(shared_labels),
#             'n_shared_features': len(shared_feats),
#             'macro_f1':      round(f1, 4),
#             'mcc':           round(mcc, 4),
#         })
#
# cross_df = pd.DataFrame(cross_results)
# cross_out = P2_OUTPUT / 'cross_dataset' / 'cross_generalization_results.csv'
# cross_df.to_csv(cross_out, index=False)
# print(f"\nCross-dataset results saved: {cross_out}")

print("Phase 2 cross-dataset generalization scaffold defined (commented out).")